# Silver province reporting areas

## Purpose

Publish the governed Option B geography approved in D-57: official provinces, NCR, independent cities that the selected PSGC hierarchy does not place under a province, and one Unknown member. Region remains the parent and reconciliation geography.

## Grain

`silver_province_reporting_area` has one governed reporting area per selected PSGC version plus key 0. `silver_province_reporting_area_boundary` has one row per reporting area and selected boundary version, including explicit missing or unsafe boundary status.


## 1. Gate inputs and select deterministic versions


In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

DECLARE OR REPLACE VARIABLE ncr_region_code STRING DEFAULT '1300000000';
DECLARE OR REPLACE VARIABLE reporting_area_rule_version STRING
    DEFAULT 'silver-province-reporting-area-v1';
DECLARE OR REPLACE VARIABLE boundary_rule_version STRING
    DEFAULT 'silver-province-reporting-area-boundary-v1';
DECLARE OR REPLACE VARIABLE boundary_srid INT DEFAULT 4326;

CREATE OR REPLACE TEMPORARY VIEW latest_reporting_area_boundary_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'boundaries'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
) AS ranked
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'STOP: latest boundary load is not safe for reporting-area construction'
)
FROM latest_reporting_area_boundary_load;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT psgc_version) <= 1
    AND COUNT(*) = COUNT(DISTINCT psgc_code),
    'STOP: selected Silver PSGC hierarchy is empty, mixed-run, or duplicated'
)
FROM `02-silver`.silver_psgc_place;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver PSGC validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_psgc_place'
  AND pipeline_run_id = (SELECT MAX(run_id) FROM `02-silver`.silver_psgc_place);

CREATE OR REPLACE TEMPORARY VIEW selected_reporting_area_versions AS
WITH psgc AS (
    SELECT
        MAX(run_id) AS psgc_run_id,
        MAX(source_snapshot_id) AS psgc_source_snapshot_id,
        MAX(source_ingest_run_id) AS psgc_source_ingest_run_id,
        MAX(psgc_version) AS psgc_version,
        MAX(source_load_ts) AS psgc_source_load_ts
    FROM `02-silver`.silver_psgc_place
),
boundary AS (
    SELECT
        MAX(source_psgc_version) AS boundary_psgc_version,
        MAX(source_boundary_version) AS boundary_version,
        MAX(_source_snapshot_id) AS boundary_source_snapshot_id,
        MAX(_ingest_run_id) AS boundary_ingest_run_id,
        MAX(_ingested_at) AS boundary_source_load_ts,
        MAX(_source_system) AS boundary_source_system
    FROM `01-bronze`.boundaries
    WHERE _source_snapshot_id = (
        SELECT snapshot_id FROM latest_reporting_area_boundary_load
    )
)
SELECT
    psgc.*,
    boundary.*,
    SHA2(CONCAT_WS(
        '||', psgc.psgc_run_id, COALESCE(psgc.psgc_version, 'NO_PUBLISHER_VERSION'),
        reporting_area_rule_version
    ), 256) AS reporting_area_run_id,
    SHA2(CONCAT_WS(
        '||', psgc.psgc_run_id, boundary.boundary_source_snapshot_id,
        boundary.boundary_ingest_run_id,
        COALESCE(boundary.boundary_version, 'NO_BOUNDARY_VERSION'),
        boundary_rule_version
    ), 256) AS boundary_run_id
FROM psgc
CROSS JOIN boundary;


## 2. Publish the governed reporting-area reference

NCR is one reporting area. A city becomes an independent-city reporting area only when the selected PSGC hierarchy gives it no province parent and its parent region is not NCR. No city is assigned to a province by name or proximity.


In [ ]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_province_reporting_area
USING DELTA
COMMENT 'Governed province reporting areas approved by D-57: official provinces, independent cities without a PSGC province parent, NCR, and Unknown.'
AS
WITH official_provinces AS (
    SELECT
        psgc_code AS psgc_reporting_area_code,
        place_name_raw AS reporting_area_name,
        'PROVINCE' AS reporting_area_type,
        region_code AS parent_region_code,
        region_name AS parent_region_name,
        psgc_version, source_snapshot_id, source_ingest_run_id, run_id AS psgc_run_id,
        source_load_ts
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'PROVINCE'
),
independent_cities AS (
    SELECT
        psgc_code AS psgc_reporting_area_code,
        place_name_raw AS reporting_area_name,
        'INDEPENDENT_CITY' AS reporting_area_type,
        region_code AS parent_region_code,
        region_name AS parent_region_name,
        psgc_version, source_snapshot_id, source_ingest_run_id, run_id AS psgc_run_id,
        source_load_ts
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'CITY'
      AND province_code IS NULL
      AND region_code <> ncr_region_code
),
ncr AS (
    SELECT
        psgc_code AS psgc_reporting_area_code,
        place_name_raw AS reporting_area_name,
        'NCR' AS reporting_area_type,
        psgc_code AS parent_region_code,
        place_name_raw AS parent_region_name,
        psgc_version, source_snapshot_id, source_ingest_run_id, run_id AS psgc_run_id,
        source_load_ts
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'REGION'
      AND psgc_code = ncr_region_code
),
geographic_areas AS (
    SELECT * FROM official_provinces
    UNION ALL
    SELECT * FROM independent_cities
    UNION ALL
    SELECT * FROM ncr
),
unknown_area AS (
    SELECT
        CAST(NULL AS STRING) AS psgc_reporting_area_code,
        'Unknown or unmapped' AS reporting_area_name,
        'UNKNOWN' AS reporting_area_type,
        CAST(NULL AS STRING) AS parent_region_code,
        CAST(NULL AS STRING) AS parent_region_name,
        selected.psgc_version,
        selected.psgc_source_snapshot_id AS source_snapshot_id,
        selected.psgc_source_ingest_run_id AS source_ingest_run_id,
        selected.psgc_run_id,
        selected.psgc_source_load_ts AS source_load_ts
    FROM selected_reporting_area_versions AS selected
),
all_areas AS (
    SELECT * FROM geographic_areas
    UNION ALL
    SELECT * FROM unknown_area
)
SELECT
    CASE
        WHEN reporting_area_type = 'UNKNOWN' THEN CAST(0 AS BIGINT)
        ELSE XXHASH64(CONCAT_WS(
            '|', 'PROVINCE_REPORTING_AREA', psgc_reporting_area_code,
            COALESCE(psgc_version, source_snapshot_id)
        ))
    END AS reporting_area_key,
    psgc_reporting_area_code,
    reporting_area_name,
    reporting_area_name AS reporting_area_short_name,
    reporting_area_type,
    parent_region_code,
    parent_region_name,
    psgc_version,
    reporting_area_type <> 'UNKNOWN' AS is_geographic_area,
    source_snapshot_id AS psgc_source_snapshot_id,
    source_ingest_run_id AS psgc_source_ingest_run_id,
    psgc_run_id,
    selected.reporting_area_run_id AS run_id,
    source_load_ts,
    reporting_area_rule_version
FROM all_areas
CROSS JOIN selected_reporting_area_versions AS selected;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT reporting_area_key)
    AND COUNT_IF(reporting_area_key = 0) = 1
    AND COUNT_IF(reporting_area_type NOT IN (
        'PROVINCE', 'INDEPENDENT_CITY', 'NCR', 'UNKNOWN'
    )) = 0
    AND COUNT_IF(
        reporting_area_type <> 'UNKNOWN' AND parent_region_code IS NULL
    ) = 0,
    'STOP: reporting-area keys, types, Unknown member, or region hierarchy are unsafe'
)
FROM `02-silver`.silver_province_reporting_area;


## 3. Parse each approved boundary once and publish explicit status


In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW reporting_area_boundary_candidates AS
SELECT
    area.reporting_area_key,
    area.psgc_reporting_area_code,
    area.reporting_area_type,
    boundary.source_feature_id,
    boundary.source_boundary_version AS boundary_version,
    boundary.source_psgc_version AS boundary_psgc_version,
    boundary._source_snapshot_id AS boundary_source_snapshot_id,
    boundary._ingest_run_id AS boundary_ingest_run_id,
    boundary._source_system AS boundary_source_system,
    boundary._ingested_at AS boundary_source_load_ts,
    TRY_TO_GEOMETRY(REGEXP_REPLACE(
        REGEXP_REPLACE(
            boundary.geometry_json,
            '"(\\[[^"]*\\])"',
            CONCAT(CHAR(36), '1')
        ),
        '"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)"',
        CONCAT(CHAR(36), '1')
    )) AS boundary_geometry
FROM `02-silver`.silver_province_reporting_area AS area
INNER JOIN `01-bronze`.boundaries AS boundary
    ON area.psgc_reporting_area_code = boundary.psgc_code
CROSS JOIN selected_reporting_area_versions AS selected
WHERE area.reporting_area_type <> 'UNKNOWN'
  AND boundary._source_snapshot_id = selected.boundary_source_snapshot_id
  AND (
      (area.reporting_area_type = 'PROVINCE'
       AND LOWER(TRIM(boundary.administrative_level)) = 'province')
      OR (area.reporting_area_type = 'INDEPENDENT_CITY'
          AND LOWER(TRIM(boundary.administrative_level)) = 'city')
      OR (area.reporting_area_type = 'NCR'
          AND LOWER(TRIM(boundary.administrative_level)) = 'region')
  );

CREATE OR REPLACE TABLE `02-silver`.silver_province_reporting_area_boundary
USING DELTA
COMMENT 'One selected boundary result per governed province reporting area with explicit missing, ambiguous, invalid, or valid status.'
AS
WITH candidate_summary AS (
    SELECT
        reporting_area_key,
        COUNT(*) AS boundary_candidate_count,
        COUNT_IF(boundary_geometry IS NOT NULL AND ST_ISVALID(boundary_geometry))
            AS valid_boundary_candidate_count
    FROM reporting_area_boundary_candidates
    GROUP BY reporting_area_key
),
single_valid_boundary AS (
    SELECT candidate.*
    FROM reporting_area_boundary_candidates AS candidate
    INNER JOIN candidate_summary AS summary
        ON candidate.reporting_area_key = summary.reporting_area_key
    WHERE summary.boundary_candidate_count = 1
      AND summary.valid_boundary_candidate_count = 1
)
SELECT
    XXHASH64(CONCAT_WS(
        '|', 'PROVINCE_REPORTING_AREA_BOUNDARY',
        CAST(area.reporting_area_key AS STRING),
        COALESCE(selected.boundary_version, 'NO_BOUNDARY_VERSION')
    )) AS boundary_key,
    area.reporting_area_key,
    area.psgc_reporting_area_code,
    area.reporting_area_type,
    CASE
        WHEN area.reporting_area_type = 'UNKNOWN' THEN 'NOT_APPLICABLE'
        WHEN COALESCE(summary.boundary_candidate_count, 0) = 0 THEN 'MISSING_BOUNDARY'
        WHEN summary.boundary_candidate_count > 1 THEN 'AMBIGUOUS_BOUNDARY'
        WHEN summary.valid_boundary_candidate_count = 0 THEN 'INVALID_BOUNDARY'
        ELSE 'VALID_BOUNDARY'
    END AS boundary_status,
    CASE
        WHEN summary.boundary_candidate_count = 1
         AND summary.valid_boundary_candidate_count = 1
            THEN ST_ASWKB(single.boundary_geometry)
    END AS boundary_geometry_wkb,
    boundary_srid AS boundary_srid,
    selected.boundary_version,
    selected.boundary_psgc_version,
    selected.boundary_source_snapshot_id,
    selected.boundary_ingest_run_id AS boundary_run_id,
    selected.boundary_source_system,
    COALESCE(summary.boundary_candidate_count, 0) AS boundary_candidate_count,
    area.psgc_run_id,
    selected.boundary_run_id AS run_id,
    GREATEST(area.source_load_ts, selected.boundary_source_load_ts) AS source_load_ts,
    boundary_rule_version
FROM `02-silver`.silver_province_reporting_area AS area
CROSS JOIN selected_reporting_area_versions AS selected
LEFT JOIN candidate_summary AS summary
    ON area.reporting_area_key = summary.reporting_area_key
LEFT JOIN single_valid_boundary AS single
    ON area.reporting_area_key = single.reporting_area_key;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_province_reporting_area)
    AND COUNT(*) = COUNT(DISTINCT reporting_area_key)
    AND COUNT(*) = COUNT(DISTINCT boundary_key)
    AND COUNT_IF(boundary_status = 'VALID_BOUNDARY'
        AND boundary_geometry_wkb IS NULL) = 0,
    'STOP: reporting-area boundary grain, keys, or valid geometry are unsafe'
)
FROM `02-silver`.silver_province_reporting_area_boundary;


# Summary

The two Silver reference tables implement D-57 without removing regions. Boundaries are parsed once, stored as WKB, and marked unsafe rather than repaired or guessed. No fixed reporting-area count is hardcoded. Databricks runtime evidence is still required.
